# Phase 4 ?????? tidy ??

`data/raw/view_by_type.txt`, `data/raw/view_by_date.txt`? ?? ?? ?? ???? ?????, Phase 4 ??? ?? `tidy_rewards.csv`? ????. `data/raw/`? ?? ????? ????.


In [1]:
from pathlib import Path
import re
import sys

import pandas as pd

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

raw_dir = project_root / 'data' / 'raw'
interim_dir = project_root / 'data' / 'interim'
processed_dir = project_root / 'data' / 'processed'
interim_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

TODAY = pd.Timestamp('2026-06-10')


In [2]:
from src.normalize import categorize, extract_magnitude, normalize_reward
from src.parse import parse_by_type, parse_by_date, parse_date_header

by_type_text = (raw_dir / 'view_by_type.txt').read_text(encoding='utf-8')
by_date_text = (raw_dir / 'view_by_date.txt').read_text(encoding='utf-8')

by_type_records = parse_by_type(by_type_text)
by_date_records = parse_by_date(by_date_text)

parsed_by_type = pd.DataFrame(by_type_records, columns=[
    'reward_raw', 'date', 'stated_cycle_days', 'is_discontinued', 'trailing'
])
parsed_by_date = pd.DataFrame(by_date_records, columns=[
    'date', 'event_context', 'is_major_event', 'no_event_week', 'reward_raw', 'reward_trailing'
])


In [3]:
def is_special(row):
    text = ' '.join(
        str(x) for x in [row.get('reward_raw'), row.get('reward_trailing'), row.get('event_context')]
        if pd.notna(x)
    )
    return '\uD0C0\uC784' in text

def time_limited(trailing):
    if pd.isna(trailing) or trailing is None:
        return None
    text = str(trailing)
    if '\uD558\uB8E8 \uC885\uC77C' in text or '\uD558\uB8E8\uC885\uC77C' in text:
        return '\uD558\uB8E8 \uC885\uC77C'
    if '\uC624\uC804' in text or '\uC624\uD6C4' in text or '~' in text:
        return text
    if re.search(r'\d+\uBD84', text):
        return text
    return None

tidy_prenorm = parsed_by_date.copy()
tidy_prenorm['is_special'] = tidy_prenorm.apply(is_special, axis=1)
tidy_prenorm['time_limited'] = tidy_prenorm['reward_trailing'].apply(time_limited)
tidy_prenorm['is_future'] = pd.to_datetime(tidy_prenorm['date']) > TODAY
tidy_prenorm = tidy_prenorm[[
    'date', 'reward_raw', 'reward_trailing', 'event_context', 'is_major_event',
    'is_special', 'time_limited', 'is_future', 'no_event_week'
]]


## Phase 4 normalization decisions

- Special-time rewards stay in benefit categories and are marked only by `is_special`.
- `??? ??` is split into ???/??? norms.
- `? ???` is split into hunting acquisition and login grant norms.
- The 2023-01 Power Up Week inline daily rewards are included in tidy output.


In [4]:
tidy_rewards = tidy_prenorm.copy()

def apply_normalize(value):
    if pd.isna(value):
        return None
    return normalize_reward(str(value))

tidy_rewards['reward_norm'] = tidy_rewards['reward_raw'].apply(apply_normalize)
tidy_rewards['category'] = tidy_rewards['reward_norm'].apply(lambda x: categorize(x) if pd.notna(x) else None)
tidy_rewards['reward_magnitude'] = tidy_rewards['reward_raw'].apply(lambda x: extract_magnitude(str(x)) if pd.notna(x) else None)
tidy_rewards = tidy_rewards[[
    'date', 'reward_raw', 'reward_trailing', 'reward_norm', 'category', 'reward_magnitude',
    'event_context', 'is_major_event', 'is_special', 'time_limited',
    'is_future', 'no_event_week'
]]


In [5]:
parsed_by_type.to_csv(interim_dir / 'parsed_by_type.csv', index=False, encoding='utf-8')
parsed_by_date.to_csv(interim_dir / 'parsed_by_date.csv', index=False, encoding='utf-8')
tidy_prenorm.to_csv(interim_dir / 'tidy_prenorm.csv', index=False, encoding='utf-8')
tidy_rewards.to_csv(processed_dir / 'tidy_rewards.csv', index=False, encoding='utf-8')

for name, base in [
    ('parsed_by_type.csv', interim_dir),
    ('parsed_by_date.csv', interim_dir),
    ('tidy_prenorm.csv', interim_dir),
    ('tidy_rewards.csv', processed_dir),
]:
    path = base / name
    print(f'{name}: exists={path.exists()} bytes={path.stat().st_size}')


parsed_by_type.csv: exists=True bytes=84712
parsed_by_date.csv: exists=True bytes=110137
tidy_prenorm.csv: exists=True bytes=125675
tidy_rewards.csv: exists=True bytes=167040


In [6]:
print('parsed_by_type rows:', len(parsed_by_type))
print('parsed_by_date rows:', len(parsed_by_date))
print('tidy_prenorm rows:', len(tidy_prenorm))
print('tidy_prenorm unique dates:', tidy_prenorm['date'].nunique())
print('tidy_prenorm min date:', tidy_prenorm['date'].min())
print('tidy_prenorm max date:', tidy_prenorm['date'].max())
print('is_future rows:', int(tidy_prenorm['is_future'].sum()))
print('future min date:', tidy_prenorm.loc[tidy_prenorm['is_future'], 'date'].min())
print('no_event_week rows:', int(tidy_prenorm['no_event_week'].sum()))
print('is_major_event ratio:', float(tidy_prenorm['is_major_event'].mean()))
print('parsed_by_type discontinued unique rewards:', parsed_by_type.loc[parsed_by_type['is_discontinued'], 'reward_raw'].nunique())

parsed_by_type rows: 1140
parsed_by_date rows: 1160
tidy_prenorm rows: 1160
tidy_prenorm unique dates: 501
tidy_prenorm min date: 2017-03-12
tidy_prenorm max date: 2026-08-30
is_future rows: 12
future min date: 2026-06-14
no_event_week rows: 30
is_major_event ratio: 0.7293103448275862
parsed_by_type discontinued unique rewards: 71


In [7]:
missing_norm = tidy_rewards['reward_norm'].isna()
unmapped = tidy_rewards[missing_norm & tidy_rewards['reward_raw'].notna()]
category_counts = tidy_rewards['category'].value_counts(dropna=False)
scored = tidy_rewards[tidy_rewards['reward_norm'].notna()]
duplicate_mask = scored.duplicated(['date', 'reward_norm'], keep=False)
duplicate_pairs = (
    scored.loc[duplicate_mask, ['date', 'reward_norm']]
    .drop_duplicates()
    .sort_values(['date', 'reward_norm'])
)
powerup = tidy_rewards[
    (tidy_rewards['date'] >= '2023-01-12')
    & (tidy_rewards['date'] <= '2023-01-18')
    & (tidy_rewards['event_context'].fillna('').str.contains('\uD30C\uC6CC\uC5C5 \uC704\uD06C'))
].copy()
dates = pd.to_datetime(tidy_rewards['date'])
non_sunday_dates = sorted(tidy_rewards.loc[dates.dt.weekday != 6, 'date'].drop_duplicates().tolist())

print('total rows:', len(tidy_rewards))
print('reward_norm missing rows:', int(missing_norm.sum()))
print('no_event_week rows:', int(tidy_rewards['no_event_week'].sum()))
print('unmapped reward_raw rows:', len(unmapped))
if len(unmapped):
    print('unmapped reward_raw values:', sorted(unmapped['reward_raw'].dropna().unique().tolist()))
print('category counts:')
print(category_counts)
print('special category rows:', int((tidy_rewards['category'] == '\uC2A4\uD398\uC15C').sum()))
print('uncategorized rows:', int((tidy_rewards['category'] == '\uBBF8\uBD84\uB958').sum()))
print('duplicate (date, reward_norm) pair count:', len(duplicate_pairs))
print('duplicate (date, reward_norm) row count:', int(duplicate_mask.sum()))
if len(duplicate_pairs):
    print(duplicate_pairs.to_string(index=False))
print('unique reward_norm count:', tidy_rewards['reward_norm'].nunique(dropna=True))
print('powerup week rows:', len(powerup))
print(powerup[['date', 'reward_raw', 'reward_trailing', 'reward_norm', 'category', 'is_special']].to_string(index=False))
print('date min:', tidy_rewards['date'].min())
print('date max:', tidy_rewards['date'].max())
print('non-sunday date count:', len(non_sunday_dates))
print('non-sunday dates:', non_sunday_dates)


total rows: 1160
reward_norm missing rows: 42
no_event_week rows: 30
unmapped reward_raw rows: 0
category counts:
category
강화      357
성장      337
사냥효율    234
수집기타    190
NaN      42
Name: count, dtype: int64
special category rows: 0
uncategorized rows: 0
duplicate (date, reward_norm) pair count: 56
duplicate (date, reward_norm) row count: 117
      date reward_norm
2017-03-12        돌발미션
2017-03-26     룬_경험치버프
2017-04-09          버닝
2017-06-25        돌발미션
2017-07-02     룬_경험치버프
2017-07-02          버닝
2017-09-03        돌발미션
2017-11-12        돌발미션
2017-12-10      몬스터컬렉션
2018-04-01         우르스
2018-04-08       파티퀘스트
2018-04-15        돌발미션
2018-04-15      몬스터컬렉션
2018-10-21      몬스터컬렉션
2018-10-28         우르스
2018-11-25        돌발미션
2018-12-23   몬스터파크_경험치
2019-04-07      몬스터컬렉션
2019-07-07        돌발미션
2019-10-20      몬스터컬렉션
2019-12-01      몬스터컬렉션
2020-04-12      몬스터컬렉션
2020-10-04      몬스터컬렉션
2020-11-15      몬스터컬렉션
2021-03-14      몬스터컬렉션
2021-08-08      몬스터컬렉션
2021-09-26      몬스터컬렉션
2021-10-17

In [8]:
date_header_count = sum(1 for line in by_date_text.splitlines() if parse_date_header(line.strip()) is not None)
date_pattern_count = len(re.findall(r'\d{4}\.\d{2}\.\d{2}', by_date_text))
s_none_count = by_date_text.count('[[S]]?? ??[[/S]]')
b_none_count = by_date_text.count('[[B]]??[[/B]]')
multi_trailing_count = int(tidy_prenorm['reward_trailing'].fillna('').str.contains(r'\)\(', regex=True).sum())

print('by_date date header count:', date_header_count)
print('by_date date pattern count:', date_pattern_count)
print('unique dates match header unique:', tidy_prenorm['date'].nunique() == parsed_by_date['date'].nunique())
print('[[S]]?? ??[[/S]] count:', s_none_count)
print('[[B]]??[[/B]] count:', b_none_count)
print('multi-parentheses trailing rows:', multi_trailing_count)
print('future rows all > today:', bool((pd.to_datetime(tidy_prenorm.loc[tidy_prenorm['is_future'], 'date']) > TODAY).all()))

by_date date header count: 502
by_date date pattern count: 503
unique dates match header unique: True
[[S]]?? ??[[/S]] count: 0
[[B]]??[[/B]] count: 0
multi-parentheses trailing rows: 1
future rows all > today: True


In [9]:
display(parsed_by_type.head())
display(parsed_by_date.head())
display(tidy_prenorm.head())

,reward_raw,date,stated_cycle_days,is_discontinued,trailing
0,"[HEXA 스탯] 메인 스탯이 5레벨 이상일 때, 메인 스탯의 강화 확률 20% 증가",2024-07-21,NaN,False,NaN
1,"[HEXA 스탯] 메인 스탯이 5레벨 이상일 때, 메인 스탯의 강화 확률 20% 증가",2024-10-13,84.0,False,NaN
2,"[HEXA 스탯] 메인 스탯이 5레벨 이상일 때, 메인 스탯의 강화 확률 20% 증가",2025-01-12,91.0,False,NaN
3,"[HEXA 스탯] 메인 스탯이 5레벨 이상일 때, 메인 스탯의 강화 확률 20% 증가",2025-04-06,84.0,False,NaN
4,"[HEXA 스탯] 메인 스탯이 5레벨 이상일 때, 메인 스탯의 강화 확률 20% 증가",2025-07-06,91.0,False,NaN


,date,event_context,is_major_event,no_event_week,reward_raw,reward_trailing
0,2017-03-12,NaN,False,False,주문의 흔적 강화 비용 30% 할인,NaN
1,2017-03-12,NaN,False,False,"(리부트, 리부트2월드 : 하루 종일 드롭률 2배)",NaN
2,2017-03-12,NaN,False,False,돌발미션 일일 클리어 가능 횟수 2배,(3회→6회)
3,2017-03-12,NaN,False,False,돌발미션 클리어 경험치 2배,NaN
4,2017-03-19,NaN,False,True,NaN,NaN


,date,reward_raw,reward_trailing,event_context,is_major_event,is_special,time_limited,is_future,no_event_week
0,2017-03-12,주문의 흔적 강화 비용 30% 할인,NaN,NaN,False,False,NaN,False,False
1,2017-03-12,"(리부트, 리부트2월드 : 하루 종일 드롭률 2배)",NaN,NaN,False,False,NaN,False,False
2,2017-03-12,돌발미션 일일 클리어 가능 횟수 2배,(3회→6회),NaN,False,False,NaN,False,False
3,2017-03-12,돌발미션 클리어 경험치 2배,NaN,NaN,False,False,NaN,False,False
4,2017-03-19,NaN,NaN,NaN,False,False,NaN,False,True


## ?? ??

- `tidy_prenorm.csv`? `by_date` ?? ?? ?? ?? ?? ?? ???? ????.
- ?? ?? ??? ??? ??? ?? ?? ??? ?? `reward_raw` ??? ????.
- ? ?? ?? ??? ?? spine ??? ?? `no_event_week=True`, `reward_raw=null` ??? ???.